# Viettel AI Race - Distill model từ bản 82x

Notebook này train NER bằng pseudo-label từ `82x_cleanroom4_k2` (bản public 38.6460), rồi chạy inference bằng model vừa train. Ý tưởng là để model học lại cấu trúc nhãn tốt hơn bản tay cũ, không nộp nguyên output teacher.

## 1. Upload gói data

Ở máy local chạy `bash colab/pack.sh`, rồi upload `artifacts/viettel_colab_data.zip` ở cell dưới.

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert zip_names, 'Bạn cần upload artifacts/viettel_colab_data.zip'

work = Path('/content/vietel_distill')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(zip_names[0]) as zf:
    zf.extractall(work)
os.chdir(work)
print('Đã giải nén vào', work)
!find data -maxdepth 2 -type d | sort
!wc -l data/ner_teacher_82x/*.jsonl

## 2. Cài thư viện và kiểm GPU

In [ ]:
!pip install -q -r requirements.txt

import torch
HAS_CUDA = torch.cuda.is_available()
print('cuda:', HAS_CUDA)
if HAS_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_CAP = torch.cuda.get_device_capability(0)
    print(GPU_NAME, 'capability=', GPU_CAP)
else:
    GPU_NAME = 'cpu'
    GPU_CAP = (0, 0)

MODEL_NAME = 'xlm-roberta-large'
TEACHER_TAG = '82x'  # đổi thành '87' để thử teacher recall rộng hơn; '88' để thử teacher chặt hơn
TEACHER_DATA = f'data/ner_teacher_{TEACHER_TAG}'
TEACHER_DIR = f'data/teacher_{TEACHER_TAG}'
TEACHER_LEXICON = f'data/kb/gt_lexicon_teacher_{TEACHER_TAG}.json'
MODEL_DEV = f'models/ner_teacher_{TEACHER_TAG}_dev'
MODEL_FINAL = f'models/ner_teacher_{TEACHER_TAG}'
PRED_OUT = f'/content/pred_teacher_{TEACHER_TAG}'
SUBMISSION_ZIP = f'/content/output_teacher_model_{TEACHER_TAG}.zip'

EPOCHS_DEV = 20
EPOCHS_FINAL = 30
BS = 4
ACCUM = 2
LR = 2e-5
# bf16 chỉ ổn trên Ampere/Ada/Hopper (capability >= 8). T4 là 7.5 nên phải dùng fp16.
PRECISION_FLAG = '--bf16' if HAS_CUDA and GPU_CAP[0] >= 8 else ('--fp16' if HAS_CUDA else '')
print('teacher:', TEACHER_TAG)
print('precision:', PRECISION_FLAG or 'fp32')

## 3. Build lại data teacher

In [ ]:
!python src/gt_lexicon.py --gt $TEACHER_DIR --out $TEACHER_LEXICON --split all
!python src/ner_data.py --gt $TEACHER_DIR --out $TEACHER_DATA

## 4. Train dev để xem model học teacher tới đâu

In [ ]:
!python src/ner_train.py --data $TEACHER_DATA --model $MODEL_NAME --out $MODEL_DEV --epochs $EPOCHS_DEV --bs $BS --accum $ACCUM --lr $LR $PRECISION_FLAG 2>&1 | tee /content/train_teacher_dev.log

## 5. Train final trên toàn bộ 100 file

In [ ]:
!python src/ner_train.py --data $TEACHER_DATA --all --model $MODEL_NAME --out $MODEL_FINAL --epochs $EPOCHS_FINAL --bs $BS --accum $ACCUM --lr $LR $PRECISION_FLAG 2>&1 | tee /content/train_teacher_final.log

## 6. Sinh output bằng model distill

In [ ]:
!rm -rf $PRED_OUT /content/output
!python src/ner_infer.py --model $MODEL_FINAL --input input --out $PRED_OUT --lexicon $TEACHER_LEXICON

## 7. Kiểm offset và tải zip nộp

In [ ]:
import json, shutil
from pathlib import Path

bad = []
for fp in sorted(Path(PRED_OUT).glob('*.json'), key=lambda p: int(p.stem)):
    raw = Path('input', f'{fp.stem}.txt').read_text(encoding='utf-8')
    ents = json.loads(fp.read_text(encoding='utf-8'))
    for i, e in enumerate(ents):
        s, t = e['position']
        if raw[s:t] != e['text']:
            bad.append((fp.name, i, e['position'], e['text'], raw[s:t]))
assert not bad, bad[:3]
print('Offset OK:', len(list(Path(PRED_OUT).glob('*.json'))), 'files')

!rm -rf /content/output
!mkdir -p /content/output
!cp $PRED_OUT/*.json /content/output/
!cd /content && zip -q -r $SUBMISSION_ZIP output
!ls -lh $SUBMISSION_ZIP
files.download(SUBMISSION_ZIP)

## 8. Probe copy/lọc theo teacher

Các zip này không train lại. Dùng để tách lỗi field và lỗi span thừa.

In [ ]:
!python src/compare_to_teacher.py --pred $PRED_OUT --teacher $TEACHER_DIR

# Copy field từ teacher đang chọn, không thêm/bớt entity.
!python src/copy_teacher_fields.py --pred $PRED_OUT --teacher $TEACHER_DIR --input input --out /content/pred_fields_selected
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_fields_selected/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_fields_selected.zip output

# Lọc bỏ span model dự đoán nhưng không có trong teacher đang chọn, rồi copy field.
!python src/filter_by_teacher.py --pred $PRED_OUT --teacher $TEACHER_DIR --input input --out /content/pred_strict_selected
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_strict_selected/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_strict_selected.zip output

# Probe riêng cho bản 85 curated khi teacher đang là 82x.
!python src/copy_teacher_fields.py --pred $PRED_OUT --teacher data/teacher_85 --input input --out /content/pred_fields85
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_fields85/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_fields85.zip output

!python src/filter_by_teacher.py --pred $PRED_OUT --teacher data/teacher_85 --input input --out /content/pred_strict85
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_strict85/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_strict85.zip output

!ls -lh /content/output_teacher_model_fields_selected.zip /content/output_teacher_model_strict_selected.zip /content/output_teacher_model_fields85.zip /content/output_teacher_model_strict85.zip
files.download('/content/output_teacher_model_fields_selected.zip')
files.download('/content/output_teacher_model_strict_selected.zip')
files.download('/content/output_teacher_model_fields85.zip')
files.download('/content/output_teacher_model_strict85.zip')

## 9. Tải weights để lưu lại run

In [ ]:
!cd models && zip -q -r /content/ner_teacher_weights.zip $(basename $MODEL_FINAL)
!ls -lh /content/ner_teacher_weights.zip
files.download('/content/ner_teacher_weights.zip')
files.download('/content/train_teacher_dev.log')
files.download('/content/train_teacher_final.log')